# Lesson 1.4 — Transforming Twists Between Frames
**Module 6 · Unit 1 · Lesson 4**

Verifies: (1) the 6x6 transform; (2) special cases (d=0 block-diagonal; R=I shifts linear only); (3) **invariance** — physical point velocities agree in both frames.

Convention D-057: $\xi=[v;\omega]$, base/world frame primary.

In [1]:
import numpy as np
checks=[]
def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def Rz(a): 
    c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1.0]])

def twist_transform(R,d):
    # maps a twist expressed in A (about A origin) to B (about B origin),
    # where B is at offset d, orientation R, relative to A (all in A coords).
    T=np.zeros((6,6))
    T[:3,:3]=R.T; T[:3,3:]=-R.T@skew(d); T[3:,3:]=R.T
    return T

# special cases
checks.append(np.allclose(twist_transform(Rz(0.0), np.zeros(3))[:3,3:],0))      # R=I,d=0 -> block diagonal
checks.append(np.allclose(twist_transform(np.eye(3),[0,0.5,0])[:3,:3],np.eye(3)))# R=I -> top-left identity
print("special-case checks:", checks[-2], checks[-1])

special-case checks: True True


## Worked example (R = I, d = (0,0.5,0))

In [2]:
xiA=np.hstack([[1.0,0,0],[0,0,2.0]])
T=twist_transform(np.eye(3),[0,0.5,0])
xiB=T@xiA
print("xi_B =", xiB, " (expect v_B=(0,0,0): B origin is the instantaneous center; w_B=(0,0,2))")
checks.append(np.allclose(xiB, [0,0,0, 0,0,2]))

xi_B = [0. 0. 0. 0. 0. 2.]  (expect v_B=(0,0,0): B origin is the instantaneous center; w_B=(0,0,2))


## Invariance: same physical point velocity computed in frame A and frame B

In [3]:
def pv(xi,r): 
    return xi[:3]+np.cross(xi[3:], r)

R=Rz(0.6); d=np.array([0.2,0.5,-0.1])
xiA=np.hstack([[0.4,-0.3,0.1],[0.0,0.2,1.1]])
xiB=twist_transform(R,d)@xiA

# a body point expressed in A coords:
pA=np.array([0.3,0.1,0.2])
vA=pv(xiA,pA)                       # its velocity, in A coords

# same point expressed in B coords, velocity in B coords, then rotated back to A:
pB=R.T@(pA-d)
vB_inB=pv(xiB,pB)
vB_inA=R@vB_inB                     # rotate B-frame velocity back into A
print("velocity via A:", vA)
print("velocity via B:", vB_inA)
checks.append(np.allclose(vA, vB_inA))

velocity via A: [0.33 0.03 0.04]
velocity via B: [0.33 0.03 0.04]


In [4]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
